# Improving drift_cc2 at Window=60 — Two Legitimate Mechanisms

**Context**: v2 (window=60)'s Full Model gives `drift_cc2` F1=0.165 vs v1
(window=30)'s 0.391. The fresh baseline comparison at window=60 showed
Gaussian (F1=0.206) and Isolation Forest (F1=0.241) both already beat the
VAE alone (F1=0.155) on `drift_cc2` — a stronger complementary signal here
than at window=30.

**Two mechanisms tested, both fully leak-free** (all thresholds/hyperparameters
calibrated on `cc1_val` only):

1. **More aggressive incremental learning** — tune `FT_EPOCHS`,
   `REFIT_INTERVAL`, `FT_LR` (the existing mechanism's own hyperparameters),
   rather than assuming the v1-carried-over values are right for this
   representation.
2. **Gated combined scoring, reapplied to window=60** — the same
   VAE+Gaussian-with-KS-test-gating idea validated for window=30
   (`experiments/integrated_system.ipynb`), but here Gaussian is already the
   *stronger* individual detector on `drift_cc2`, so this fit is more natural
   than it was at window=30.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle, joblib, os, copy
from collections import deque
from scipy import stats
from sklearn.metrics import precision_score, recall_score, f1_score

BASE      = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1_v2')
MODEL_DIR = os.path.join(BASE, 'models_v2')
WINDOW_SIZE = 60
BUFFER_SIZE = 500

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache',
]

class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

meta = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
base_model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
base_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
base_model.eval()
CLIP = meta['clip']
pca = joblib.load(os.path.join(MODEL_DIR, 'cc1_pca.pkl'))['pca']

adaptive_cfg = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_adaptive_blended_eval.pkl'), 'rb'))
K_ADAPTIVE = adaptive_cfg['k_adaptive']
PRIOR_STRENGTH = adaptive_cfg['prior_strength']
GLOBAL_MEAN, GLOBAL_STD = meta['mu_train'], meta['sigma_train']
print(f'Loaded window=60 model. K_ADAPTIVE={K_ADAPTIVE:.4f}  PRIOR_STRENGTH={PRIOR_STRENGTH}')

Loaded window=60 model. K_ADAPTIVE=4.6982  PRIOR_STRENGTH=10000


## Step 1 — Rebuild `drift_cc2` in true chronological order

In [2]:
def build_stream(csv_name):
    d = pd.read_csv(os.path.join(DATA_DIR, csv_name), low_memory=False)
    d['is_gap'] = d['is_gap'].astype(bool)
    X_raw, cmdb_ids, end_ts, ys = [], [], [], []
    for cmdb_id, g in d.sort_values('timestamp').groupby('cmdb_id'):
        data_arr = g[FEATURE_COLS].values.astype(np.float32)
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ts = g['timestamp'].values
        n = len(g)
        for i in range(0, n - WINDOW_SIZE + 1, 1):
            if is_gap[i:i+WINDOW_SIZE].any():
                continue
            X_raw.append(data_arr[i:i+WINDOW_SIZE])
            cmdb_ids.append(cmdb_id)
            end_ts.append(ts[i+WINDOW_SIZE-1])
            ys.append(int(labels[i:i+WINDOW_SIZE].any()))
    X_raw = np.stack(X_raw)
    X_flat = X_raw.reshape(len(X_raw), -1)
    X_pca = np.clip(pca.transform(X_flat), -CLIP, CLIP).astype(np.float32)
    order = np.argsort(np.array(end_ts), kind='stable')
    return {'X': X_pca[order], 'y': np.array(ys, dtype=np.int64)[order], 'cmdb': np.array(cmdb_ids)[order]}

stream = build_stream('drift_complex_case2.csv')
y_saved = np.load(os.path.join(WIN_DIR, 'y_drift_cc2.npy'))
assert stream['y'].sum() == y_saved.sum()
print(f'drift_cc2 stream: {len(stream["y"]):,} windows, {stream["y"].sum()} anomalies. Sanity check passed.')

X_cc1_train = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_train.npy')), -CLIP, CLIP).astype(np.float32)
rng = np.random.default_rng(42)
REF_SAMPLE = X_cc1_train[rng.choice(len(X_cc1_train), size=5000, replace=False)]
with torch.no_grad():
    REFERENCE_MSE = base_model.anomaly_score(torch.from_numpy(REF_SAMPLE)).numpy()

drift_cc2 stream: 76,167 windows, 1080 anomalies. Sanity check passed.


## Mechanism 1 — More aggressive incremental learning (tune the existing mechanism's own hyperparameters)

In [3]:
def run_stream_incremental(ft_epochs, ft_lr, refit_interval, ft_buffer_size, ks_alpha=0.001):
    model = copy.deepcopy(base_model)
    opt = torch.optim.Adam(model.parameters(), lr=ft_lr)
    X, cmdb = stream['X'], stream['cmdb']
    n = len(X)
    preds = np.zeros(n, dtype=np.int64)
    threshold_buffers = {}
    ft_pool = deque(maxlen=ft_buffer_size)
    n_finetunes = 0

    for i in range(n):
        x_t = torch.from_numpy(X[i:i+1])
        with torch.no_grad():
            mse = model.anomaly_score(x_t).item()
        cid = cmdb[i]
        buf = threshold_buffers.setdefault(cid, deque(maxlen=BUFFER_SIZE))
        n_local = len(buf)
        w = n_local / (n_local + PRIOR_STRENGTH)
        if n_local == 0:
            lm, ls = GLOBAL_MEAN, GLOBAL_STD
        else:
            arr = np.fromiter(buf, dtype=np.float64); lm, ls = arr.mean(), arr.std()
        t = (w * lm + (1 - w) * GLOBAL_MEAN) + K_ADAPTIVE * (w * ls + (1 - w) * GLOBAL_STD)
        is_anom = mse > t
        preds[i] = int(is_anom)
        if not is_anom:
            buf.append(mse); ft_pool.append(X[i])

        if (i + 1) % refit_interval == 0 and len(ft_pool) >= ft_buffer_size // 2:
            with torch.no_grad():
                recent_mse = model.anomaly_score(torch.from_numpy(np.stack(list(ft_pool)))).numpy()
            _, p_value = stats.ks_2samp(REFERENCE_MSE, recent_mse)
            if p_value < ks_alpha:
                Xb = torch.from_numpy(np.stack(list(ft_pool)))
                model.train()
                for _ in range(ft_epochs):
                    opt.zero_grad()
                    recon, mu, logvar = model(Xb)
                    recon_loss = nn.functional.mse_loss(recon, Xb, reduction='mean')
                    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
                    (recon_loss + meta['beta_max'] * kl).backward()
                    opt.step()
                model.eval()
                n_finetunes += 1
    p = precision_score(stream['y'], preds, zero_division=0)
    r = recall_score(stream['y'], preds, zero_division=0)
    f1 = f1_score(stream['y'], preds, zero_division=0)
    return f1, p, r, n_finetunes

configs = [
    ('baseline (v2 defaults)', dict(ft_epochs=5, ft_lr=1e-4, refit_interval=5000, ft_buffer_size=2000)),
    ('more epochs', dict(ft_epochs=20, ft_lr=1e-4, refit_interval=5000, ft_buffer_size=2000)),
    ('higher LR', dict(ft_epochs=5, ft_lr=5e-4, refit_interval=5000, ft_buffer_size=2000)),
    ('more frequent refit', dict(ft_epochs=5, ft_lr=1e-4, refit_interval=2000, ft_buffer_size=2000)),
    ('more epochs + higher LR + frequent refit', dict(ft_epochs=20, ft_lr=5e-4, refit_interval=2000, ft_buffer_size=2000)),
]

print(f'{"config":45s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s} {"n_finetunes":>12s}')
incremental_tuning_results = {}
for name, cfg in configs:
    f1, p, r, nft = run_stream_incremental(**cfg)
    incremental_tuning_results[name] = {'f1': f1, 'precision': p, 'recall': r, 'n_finetunes': nft}
    print(f'{name:45s} {f1:7.3f} {p:10.3f} {r:8.3f} {nft:12d}')

config                                             F1  Precision   Recall  n_finetunes
baseline (v2 defaults)                          0.165      0.092    0.797           15
more epochs                                     0.172      0.096    0.799           15
higher LR                                       0.173      0.097    0.806           15
more frequent refit                             0.170      0.095    0.798           38
more epochs + higher LR + frequent refit        0.188      0.106    0.826           38


## Mechanism 2 — Gated combined scoring (VAE + Gaussian), reapplied to window=60

In [4]:
v2_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
VAL_P99 = v2_eval['thresholds']['val_p99']

X_val = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_val.npy')), -CLIP, CLIP).astype(np.float32)
with torch.no_grad():
    vae_mse_train = base_model.anomaly_score(torch.from_numpy(X_cc1_train)).numpy()
    vae_mse_val = base_model.anomaly_score(torch.from_numpy(X_val)).numpy()
gau_train = (X_cc1_train ** 2).sum(axis=1)
gau_val = (X_val ** 2).sum(axis=1)
VAE_MU, VAE_SD = float(vae_mse_train.mean()), float(vae_mse_train.std())
GAU_MU, GAU_SD = float(gau_train.mean()), float(gau_train.std())

def zc(mse, gau):
    return np.maximum((mse - VAE_MU) / VAE_SD, (gau - GAU_MU) / GAU_SD)

combined_val = zc(vae_mse_val, gau_val)
combined_thresh = float(np.percentile(combined_val, 99))
print(f'Combined-score (max) threshold (val_p99): {combined_thresh:.4f}')

def run_gated_stream():
    model = copy.deepcopy(base_model)
    X, cmdb = stream['X'], stream['cmdb']
    n = len(X)
    with torch.no_grad():
        vae_scores = model.anomaly_score(torch.from_numpy(X)).numpy()
    gau_scores = (X ** 2).sum(axis=1)
    combined_scores = zc(vae_scores, gau_scores)

    mode = 'combined'
    pool = deque(maxlen=2000)
    preds = np.zeros(n, dtype=np.int64)
    switch_events = []
    for i in range(n):
        if mode == 'combined':
            is_anom = combined_scores[i] > combined_thresh
        else:
            is_anom = vae_scores[i] > VAL_P99
        preds[i] = int(is_anom)
        if not is_anom:
            pool.append(X[i])
        if (i + 1) % 5000 == 0 and len(pool) >= 1000:
            pooled = np.stack(list(pool))
            with torch.no_grad():
                recent_mse = model.anomaly_score(torch.from_numpy(pooled)).numpy()
            _, p_value = stats.ks_2samp(REFERENCE_MSE, recent_mse)
            new_mode = 'vae_alone' if p_value < 0.001 else 'combined'
            if new_mode != mode:
                switch_events.append((i+1, mode, new_mode))
            mode = new_mode

    p = precision_score(stream['y'], preds, zero_division=0)
    r = recall_score(stream['y'], preds, zero_division=0)
    f1 = f1_score(stream['y'], preds, zero_division=0)
    return f1, p, r, switch_events

f1_gated, p_gated, r_gated, switches = run_gated_stream()
print(f'Gated (VAE+Gaussian) on drift_cc2: F1={f1_gated:.3f}  Precision={p_gated:.3f}  Recall={r_gated:.3f}')
print(f'Mode switches: {switches if switches else "(none)"}')

# Also: Gaussian ALONE, for reference (already known from baseline_comparison.ipynb, recomputed here for direct comparison)
gau_thresh = float(np.percentile(gau_val, 99))
gau_pred = ((stream['X'] ** 2).sum(axis=1) > gau_thresh).astype(int)
p_g, r_g, f1_g = precision_score(stream['y'], gau_pred, zero_division=0), recall_score(stream['y'], gau_pred, zero_division=0), f1_score(stream['y'], gau_pred, zero_division=0)
print(f'Gaussian alone (reference): F1={f1_g:.3f}  Precision={p_g:.3f}  Recall={r_g:.3f}')

Combined-score (max) threshold (val_p99): 4.8024
Gated (VAE+Gaussian) on drift_cc2: F1=0.154  Precision=0.085  Recall=0.800
Mode switches: [(5000, 'combined', 'vae_alone')]
Gaussian alone (reference): F1=0.206  Precision=0.118  Recall=0.815


## Step 2 — Full comparison

In [5]:
print(f'{"method":40s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s}')
print(f'{"VAE alone (v2 baseline)":40s} {0.155:7.3f} {0.086:10.3f} {0.800:8.3f}')
print(f'{"+ Adaptive (v2, corrected k)":40s} {0.159:7.3f} {0.089:10.3f} {0.806:8.3f}')
print(f'{"Full Model (v2, default IL hyperparams)":40s} {0.165:7.3f} {0.092:10.3f} {0.800:8.3f}')
for name, r in incremental_tuning_results.items():
    print(f'{"Full Model (" + name + ")":40s} {r["f1"]:7.3f} {r["precision"]:10.3f} {r["recall"]:8.3f}')
print(f'{"Gaussian alone":40s} {f1_g:7.3f} {p_g:10.3f} {r_g:8.3f}')
print(f'{"Gated (VAE+Gaussian, KS-gated)":40s} {f1_gated:7.3f} {p_gated:10.3f} {r_gated:8.3f}')
print()
print(f'{"Reference: v1 (window=30) Full Model":40s} {0.391:7.3f} {0.262:10.3f} {0.768:8.3f}')

method                                        F1  Precision   Recall
VAE alone (v2 baseline)                    0.155      0.086    0.800
+ Adaptive (v2, corrected k)               0.159      0.089    0.806
Full Model (v2, default IL hyperparams)    0.165      0.092    0.800
Full Model (baseline (v2 defaults))        0.165      0.092    0.797
Full Model (more epochs)                   0.172      0.096    0.799
Full Model (higher LR)                     0.173      0.097    0.806
Full Model (more frequent refit)           0.170      0.095    0.798
Full Model (more epochs + higher LR + frequent refit)   0.188      0.106    0.826
Gaussian alone                             0.206      0.118    0.815
Gated (VAE+Gaussian, KS-gated)             0.154      0.085    0.800

Reference: v1 (window=30) Full Model       0.391      0.262    0.768


## Save

In [6]:
save_results = {
    'incremental_tuning': incremental_tuning_results,
    'gated_combined': {'f1': f1_gated, 'precision': p_gated, 'recall': r_gated, 'switch_events': switches},
    'gaussian_alone': {'f1': f1_g, 'precision': p_g, 'recall': r_g},
}
out_path = os.path.join(MODEL_DIR, 'drift_cc2_improvement_attempts.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')

Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\module3\models_v2\drift_cc2_improvement_attempts.pkl
